# Getting started with ChromaDB

In the following exercises, you'll use a vector database to embed and query 1000 films
and TV shows from the Netflix dataset introduced in the video.
The goal will be to use this data to generate recommendations based on a search query.
To get started, you'll create the database and collection to store the data.

In [2]:
import os
from typing import cast

import chromadb
from chromadb.api.types import Embeddable, EmbeddingFunction
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction
from dotenv import load_dotenv
from file_helper import get_file_path

load_dotenv()
DB_PATH = get_file_path("chroma")
model_name = "text-text-embedding-3-small"
collection_name = "netflix_titles"

# Create a persistent client
client = chromadb.PersistentClient(path=DB_PATH)

# Create OpenAI embedding function
embedding_function = OpenAIEmbeddingFunction(
    model_name=model_name,
    api_key=os.getenv("OPENAI_API_KEY"),
)

# Create a netflix_title collection using the OpenAI Embedding function
collection = client.get_or_create_collection(
    name=collection_name,
    embedding_function=cast(EmbeddingFunction[Embeddable], embedding_function),
)

# List the collections
print(client.list_collections())

[Collection(name=netflix_titles)]


# Adding data to the collection

Time to add those Netflix films and TV shows to your collection!
You've been provided with a list of document IDs and texts,
stored in ids and documents, respectively,
which have been extracted from netflix_titles.csv using the following code:

```python
ids = []
documents = []

with open('netflix_titles.csv') as csvfile:
  reader = csv.DictReader(csvfile)
  for i, row in enumerate(reader):
    ids.append(row['show_id'])
    text = f"Title: {row['title']} ({row['type']})\nDescription: {row['description']}\nCategories: {row['listed_in']}"
    documents.append(text)
```


As an example of what information will be embedded, here's the first document from documents:
Title: Dick Johnson Is Dead (Movie)
Description: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.
Categories: Documentaries

In [3]:
from chromadb_helper import get_or_create_collection
from DATA import documents, ids

collection = get_or_create_collection(collection_name)

# Add the documents and IDs to the collection
collection.add(ids=ids, documents=documents)

# Print the collection size and IDs to the collection
print(f"No. of documents: {collection.count()}")
print(f"First ten docuemnts: {collection.peek()}")

No. of documents: 1000
First ten docuemnts: {'ids': ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10'], 'embeddings': array([[ 0.01786804,  0.04470825, -0.02833557, ...,  0.02737427,
        -0.00348282, -0.02635193],
       [-0.00907135,  0.08422852, -0.04373169, ...,  0.02339172,
         0.03695678, -0.04510498],
       [-0.01194   ,  0.04888916, -0.04345703, ..., -0.00445557,
         0.00289917, -0.04797363],
       ...,
       [-0.02874756,  0.05023193, -0.02987671, ...,  0.03793335,
        -0.01844788, -0.02403259],
       [ 0.00711823,  0.02452087, -0.02056885, ...,  0.01651001,
         0.01795959, -0.05142212],
       [ 0.00342941,  0.06646728, -0.03991699, ...,  0.01657104,
         0.01902771,  0.00060368]], shape=(10, 1536)), 'documents': ['Title: Dick Johnson Is Dead (Movie)\nDescription: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.\nCategories: Document

# Querying the Netflix collection

Now that you've created and populated the netflix_titles collection, it's time to query it!

As a first trial, you'll use it to provide recommendations for films and TV shows
about dogs to one of your colleagues who loves dogs!

In [4]:
from chromadb_helper import get_or_create_collection

# Retrieve the netflix_titles collection
collection = get_or_create_collection(name=collection_name)

# Query the collection for "films about dogs"
result = collection.query(
    query_texts=["films about dogs"],
    n_results=3,
)

print(result)

{'ids': [['s95', 's830', 's500']], 'embeddings': None, 'documents': [['Title: Show Dogs (Movie)\nDescription: A rough and tough police dog must go undercover with an FBI agent as a prim and proper pet at a dog show to save a baby panda from an illegal sale.\nCategories: Children & Family Movies, Comedies', 'Title: Dog Gone Trouble (Movie)\nDescription: The privileged life of a pampered dog named Trouble is turned upside-down when he gets lost and must learn to survive on the big-city streets.\nCategories: Children & Family Movies, Comedies', 'Title: Dogs (TV Show)\nDescription: These six intimate stories explore the abiding emotional bonds that form between dogs and their caregivers, no matter the circumstances.\nCategories: Docuseries']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None]], 'distances': [[0.425886869430542, 0.4364742040634155, 0.4478273391723633]]}


# Updating and deleting items from a collection

Updating and deleting items from a collection
Just because the documents have been stored away in a vector database,
that doesn't mean that you can't make changes to add to the collection or update existing items.

In this exercise, you've been provided with two new Netflix titles stored in new_data:

[{"id": "s1001", "document": "Title: Cats & Dogs (Movie)\nDescription: A look at the top-secret, high-tech espionage war going on between cats and dogs, of which their human owners are blissfully unaware."},
 {"id": "s6884", "document": 'Title: Goosebumps 2: Haunted Halloween (Movie)\nDescription: Three teens spend their Halloween trying to stop a magical book, which brings characters from the "Goosebumps" novels to life.\nCategories: Children & Family Movies, Comedies'}]

You'll either add or update these IDs in the database depending on whether they're already present in the collection.

In [5]:
from chromadb_helper import get_or_create_collection

new_data = [
    {
        "id": "s1001",
        "document": "Title: Cats & Dogs (Movie)\nDescription: A look at the top-secret, high-tech espionage war going on between cats and dogs, of which their human owners are blissfully unaware.",
    },
    {
        "id": "s6884",
        "document": 'Title: Goosebumps 2: Haunted Halloween (Movie)\nDescription: Three teens spend their Halloween trying to stop a magical book, which brings characters from the "Goosebumps" novels to life.\nCategories: Children & Family Movies, Comedies',
    },
]

collection = get_or_create_collection(name=collection_name)

# Update or add the new documents
collection.upsert(
    ids=[new_data[0]["id"], new_data[1]["id"]],
    documents=[new_data[0]["document"], new_data[1]["document"]],
)

# Delete the item with ID "s95"
collection.delete(ids=["s95"])

result = collection.query(query_texts=["films about dogs"], n_results=3)
print(result)

{'ids': [['s830', 's500', 's1001']], 'embeddings': None, 'documents': [['Title: Dog Gone Trouble (Movie)\nDescription: The privileged life of a pampered dog named Trouble is turned upside-down when he gets lost and must learn to survive on the big-city streets.\nCategories: Children & Family Movies, Comedies', 'Title: Dogs (TV Show)\nDescription: These six intimate stories explore the abiding emotional bonds that form between dogs and their caregivers, no matter the circumstances.\nCategories: Docuseries', 'Title: Cats & Dogs (Movie)\nDescription: A look at the top-secret, high-tech espionage war going on between cats and dogs, of which their human owners are blissfully unaware.']], 'uris': None, 'included': ['metadatas', 'documents', 'distances'], 'data': None, 'metadatas': [[None, None, None]], 'distances': [[0.4364742040634155, 0.4478273391723633, 0.4635791778564453]]}


# Querying with multiple texts

In many cases, you'll want to query the vector database using multiple query texts.
Recall that these query texts are embedded using the same embedding function as
when the documents were added.

In this exercise, you'll use the documents from two IDs
in the netflix_titles collection to query the rest of the collection,
returning the most similar results as recommendations.

In [6]:
from chromadb_helper import get_or_create_collection

collection = get_or_create_collection(name=collection_name)

reference_ids = ["s999", "s1000"]

# Retrieve the documents for the reference_ids
reference_texts = collection.get(ids=reference_ids)["documents"]

# Query using reference_texts
result = collection.query(query_texts=reference_texts, n_results=3)
print(result)

{'ids': [['s999', 's207', 's893'], ['s1000', 's947', 's553']], 'embeddings': None, 'documents': [['Title: Searching For Sheela (Movie)\nDescription: Journalists and fans await Ma Anand Sheela as the infamous former Rajneesh commune’s spokesperson returns to India after decades for an interview tour.\nCategories: Documentaries, International Movies', 'Title: LSD: Love, Sex Aur Dhokha (Movie)\nDescription: This provocative drama examines how the voyeuristic nature of modern society affects three unusual couples in Northern India.\nCategories: Dramas, Independent Movies, International Movies', 'Title: Tottaa Pataaka Item Maal (Movie)\nDescription: Exasperated with living in perpetual fear for their safety, four women kidnap a man to show him the realities of being female in Delhi.\nCategories: Dramas, Independent Movies, International Movies'], ['Title: Stowaway (Movie)\nDescription: A three-person crew on a mission to Mars faces an impossible choice when an unplanned passenger jeopardize

# Adding metadata

In [7]:
import csv
import os

from chromadb_helper import get_or_create_collection
from DATA import ids
from file_helper import get_file_path

csv_path = get_file_path("netflix_titles.csv")

metadatas = []

with open(csv_path, encoding="utf-8") as csvfile:
    reader = csv.DictReader(csvfile)
    for i, row in enumerate(reader):
        metadatas.append(
            {
                "type": row["type"],
                "release_year": int(row["release_year"]),
                "rating": str(row["rating"]),
            },
        )
        if i == len(ids) - 1:
            break

collection = get_or_create_collection(name=collection_name)

collection.update(ids=ids, metadatas=metadatas)
print(f"First ten documents: {collection.peek()}")


First ten documents: {'ids': ['s1', 's2', 's3', 's4', 's5', 's6', 's7', 's8', 's9', 's10'], 'embeddings': array([[ 0.01786804,  0.04470825, -0.02833557, ...,  0.02737427,
        -0.00348282, -0.02635193],
       [-0.00907135,  0.08422852, -0.04373169, ...,  0.02339172,
         0.03695678, -0.04510498],
       [-0.01194   ,  0.04888916, -0.04345703, ..., -0.00445557,
         0.00289917, -0.04797363],
       ...,
       [-0.02874756,  0.05023193, -0.02987671, ...,  0.03793335,
        -0.01844788, -0.02403259],
       [ 0.00711823,  0.02452087, -0.02056885, ...,  0.01651001,
         0.01795959, -0.05142212],
       [ 0.00342941,  0.06646728, -0.03991699, ...,  0.01657104,
         0.01902771,  0.00060368]], shape=(10, 1536)), 'documents': ['Title: Dick Johnson Is Dead (Movie)\nDescription: As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.\nCategories: Documentaries', 'Title: Blood &

# Filtering using metadata

Having metadata available to use in the database can unlock the ability
to more easily filter results based on additional conditions.
Imagine that the film recommendations you've be creating could access
the user's set preferences and use those to further filter the results.

In this exercise, you'll be using additional metadata
to filter your Netflix film recommendations.
The netflix_titles collection has been updated to add metadatas to each title,
including the 'rating', the age rating given to the title, and 'release_year',
the year the title was initially released.

Here's a preview of an updated item:

{'ids': ['s999'],
 'embeddings': None,
 'metadatas': [{'rating': 'TV-14', 'release_year': 2021}],
 'documents': ['Title: Searching For Sheela (Movie)\nDescription: Journalists and fans await Ma Anand Sheela as the infamous former Rajneesh commune’s spokesperson returns to India after decades for an interview tour.\nCategories: Documentaries, International Movies'],
 'uris': None,
 'data': None}

In [8]:
from chromadb_helper import get_or_create_collection

collection = get_or_create_collection(name=collection_name)

reference_texts = ["children's story about a car", "lions"]

# Query two results using reference_texts
result = collection.query(
    query_texts=reference_texts,
    n_results=2,
    # Filter for titles with a G rating released before 2019
    where={
        "$and": [
            {"rating": {"$eq": "G"}},
            {"release_year": {"$lt": 2019}},
        ]
    },
)

print(result["documents"])

[['Title: The Prince Who Turns into a Frog (TV Show)\nDescription: Before his wedding, a cold CEO is hit by a car and awakens a changed man. As he and the driver fall in love, more changes appear on the road ahead.\nCategories: International TV Shows, Romantic TV Shows, TV Comedies', "Title: The Mirror Has Two Faces (Movie)\nDescription: Tired of being single, middle-aged professor Rose Morgan accepts her colleague's proposal – but her colleague only wants a platonic marriage.\nCategories: Comedies, Dramas, Romantic Movies"], ['Title: Brick Mansions (Movie)\nDescription: An undercover police detective partners with an ex-convict to take down a drug kingpin amid grime and crime in a walled-off Detroit housing project.\nCategories: Action & Adventure, International Movies', 'Title: The Interview (Movie)\nDescription: Seth Rogen and James Franco star in this provocative comedy about two journalists recruited by the CIA after they arrange an interview with Kim Jong-un.\nCategories: Action 